# Week 5 — Additional Models: Decision Tree and Random Forest

In this notebook, I tested Decision Tree and Random Forest regression models to predict California residential single-family property close prices. The goal is to compare these tree-based models against the Week 4 Linear Regression baseline.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
train_df = pd.read_csv("train_residential_single_family_week3.csv")
test_df = pd.read_csv("test_residential_single_family_week3.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (129867, 57)
Test shape: (12024, 57)


In [3]:
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "YearBuilt",
    "GarageSpaces",
    "ParkingTotal",
    "Latitude",
    "Longitude",
    "Stories",
    "AssociationFee",
    "MainLevelBedrooms"
]

categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring",
    "AttachedGarageYN",
    "Levels",
    "PoolPrivateYN",
    "NewConstructionYN",
    "ViewYN",
    "FireplaceYN"
]

In [4]:
reference_only_cols = ["OriginalListPrice", "ListPrice", "DaysOnMarket", "price_per_sqft"]
reference_only_flags = [
    f"{c}_missing_flag" 
    for c in reference_only_cols 
    if f"{c}_missing_flag" in train_df.columns
]

target_derived_cols = ["log_ClosePrice"]

exclude_from_model = reference_only_cols + reference_only_flags + target_derived_cols

feature_numeric = numeric_features.copy()

for col in ["home_age", "log_LivingArea", "log_LotSizeSquareFeet"]:
    if col in train_df.columns:
        feature_numeric.append(col)

missing_flag_cols = [col for col in train_df.columns if col.endswith("_missing_flag")]
feature_numeric += missing_flag_cols

feature_numeric = list(dict.fromkeys([col for col in feature_numeric if col in train_df.columns]))
feature_categorical = [col for col in categorical_features if col in train_df.columns]

feature_numeric = [col for col in feature_numeric if col not in exclude_from_model]
feature_categorical = [col for col in feature_categorical if col not in exclude_from_model]

print("Number of numeric predictors:", len(feature_numeric))
print("Number of categorical predictors:", len(feature_categorical))

print("\nNumerical predictors:")
print(feature_numeric)

print("\nCategorical predictors:")
print(feature_categorical)

Number of numeric predictors: 32
Number of categorical predictors: 11

Numerical predictors:
['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'LotSizeAcres', 'LotSizeArea', 'YearBuilt', 'GarageSpaces', 'ParkingTotal', 'Latitude', 'Longitude', 'Stories', 'AssociationFee', 'MainLevelBedrooms', 'home_age', 'log_LivingArea', 'log_LotSizeSquareFeet', 'LivingArea_missing_flag', 'BedroomsTotal_missing_flag', 'BathroomsTotalInteger_missing_flag', 'LotSizeSquareFeet_missing_flag', 'LotSizeAcres_missing_flag', 'LotSizeArea_missing_flag', 'YearBuilt_missing_flag', 'GarageSpaces_missing_flag', 'ParkingTotal_missing_flag', 'Latitude_missing_flag', 'Longitude_missing_flag', 'Stories_missing_flag', 'AssociationFee_missing_flag', 'MainLevelBedrooms_missing_flag', 'home_age_missing_flag']

Categorical predictors:
['CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict', 'Flooring', 'AttachedGarageYN', 'Levels', 'PoolPrivateYN', 'NewConstructionYN', 'ViewYN', 'Fireplace

In [5]:
X_train_raw = train_df[feature_numeric + feature_categorical].copy()
X_test_raw = test_df[feature_numeric + feature_categorical].copy()

y_train = train_df["ClosePrice"].copy()
y_test = test_df["ClosePrice"].copy()

print("X_train_raw shape:", X_train_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train_raw shape: (129867, 43)
X_test_raw shape: (12024, 43)
y_train shape: (129867,)
y_test shape: (12024,)


In [6]:
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer

for col in feature_categorical:
    # converts everything to string 
    X_train_raw[col] = X_train_raw[col].astype(str)
    X_test_raw[col] = X_test_raw[col].astype(str)

try:
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=True
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse=True
    )


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", feature_numeric), # Don't change these numeric column, just keep them
        ("cat", encoder, feature_categorical) # onehotencode for categorial features
    ],
    remainder='drop'
)

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

In [9]:
print("Raw X_train shape:", X_train_raw.shape)
print("Processed X_train shape:", X_train_processed.shape)
print("Raw X_test shape:", X_test_raw.shape)
print("Processed X_test shape:", X_test_processed.shape)
print("Data type:", type(X_train_processed))

Raw X_train shape: (129867, 43)
Processed X_train shape: (129867, 3895)
Raw X_test shape: (12024, 43)
Processed X_test shape: (12024, 3895)
Data type: <class 'scipy.sparse._csr.csr_matrix'>


#### **Train Decision Tree**

In [10]:
from sklearn.tree import DecisionTreeRegressor

decision_tree = DecisionTreeRegressor(
    max_depth=12,
    min_samples_leaf=50,
    random_state=42
)

In [11]:
decision_tree.fit(X_train_processed, y_train)

dt_pred = decision_tree.predict(X_test_processed)


print("Decision Tree model trained.")
print("Number of predictions:", len(dt_pred))
print("First 5 predictions:", dt_pred[:5])

Decision Tree model trained.
Number of predictions: 12024
First 5 predictions: [ 897495.73401885  496401.5952915  4070202.09259259  496401.5952915
  823228.99855448]


In [12]:
dt_r2 = r2_score(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

print("Decision Tree Results")
print("R²:", dt_r2)
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)

Decision Tree Results
R²: -0.22783392239825506
MAE: 538131.2198064434
RMSE: 1859419.2911617595


The Decision Tree model performed worse than the Linear Regression baseline. Its test R² was -0.228, compared with the baseline R² of 0.287. A negative R² means the model performed worse than simply predicting the average close price for every property. The Decision Tree also had a higher MAE and RMSE than the baseline, suggesting that this single tree did not generalize well to the May 2026 test set. This may be because a single Decision Tree can be sensitive to the training data and may overfit or create unstable prediction rules.

#### **Random Forest**

In [13]:
from sklearn.ensemble import RandomForestRegressor

random_forest = RandomForestRegressor(
    n_estimators=50,
    max_depth=18,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)


random_forest.fit(X_train_processed, y_train)

rf_pred = random_forest.predict(X_test_processed)

print("Random Forest model trained.")
print("Number of predictions:", len(rf_pred))
print("First 5 predictions:", rf_pred[:5])

Random Forest model trained.
Number of predictions: 12024
First 5 predictions: [ 860873.95586237  566546.94620869 4831355.99435632  363280.90841545
 1439676.64788472]


In [14]:
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print("Random Forest Results")
print("R²:", rf_r2)
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)

Random Forest Results
R²: 0.0340345465313624
MAE: 371848.9678733415
RMSE: 1649256.9231636084


In [15]:
baseline_r2 = 0.2871733970446031
baseline_mae = 452569.1626068585
baseline_rmse = 1416770.3784853774

dt_r2 = -0.22783392239825506
dt_mae = 538131.2198064434
dt_rmse = 1859419.2911617595

rf_r2 = 0.0340345465313624
rf_mae = 371848.9678733415
rf_rmse = 1649256.9231636084

model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression Baseline",
        "Decision Tree",
        "Random Forest"
    ],
    "Training Window": [
        "12 months",
        "12 months",
        "12 months"
    ],
    "Test Month": [
        "2026-05",
        "2026-05",
        "2026-05"
    ],
    "R2": [
        baseline_r2,
        dt_r2,
        rf_r2
    ],
    "MAE": [
        baseline_mae,
        dt_mae,
        rf_mae
    ],
    "RMSE": [
        baseline_rmse,
        dt_rmse,
        rf_rmse
    ]
})

model_comparison.sort_values("R2", ascending=False)

,Model,Training Window,Test Month,R2,MAE,RMSE
0,Linear Regression Baseline,12 months,2026-05,0.287173,452569.162607,1.416770e+06
2,Random Forest,12 months,2026-05,0.034035,371848.967873,1.649257e+06
1,Decision Tree,12 months,2026-05,-0.227834,538131.219806,1.859419e+06


The Linear Regression baseline had the highest R² score at 0.287, meaning it explained the most variation in close prices among the three models. The Decision Tree performed the worst, with a negative R² score, suggesting that a single tree did not generalize well to the May 2026 test set.

The Random Forest improved over the single Decision Tree and had the lowest MAE at about $371,849, meaning it had the smallest average dollar error. However, its R² score was only 0.034 and its RMSE was higher than the Linear Regression baseline. This suggests that although Random Forest may make better average predictions for many homes, it still struggled with overall price variation and large errors on expensive or unusual properties.

Overall, the Week 4 Linear Regression baseline remained the strongest model by R². The Week 5 tree-based models did not improve the main evaluation metric, but the Random Forest result shows potential because it reduced the average absolute error compared with the baseline.

## Conclusion

For Week 5, I trained and evaluated Decision Tree and Random Forest regressors to compare against the updated Week 4 Linear Regression baseline. All models used the same 12-month training window and were evaluated on the May 2026 test set.

The Linear Regression baseline achieved the highest R² score at 0.287. The Decision Tree model performed worse than the baseline, with a negative R² score of -0.228, indicating that a single tree did not generalize well to the test month. The Random Forest model improved over the single Decision Tree and achieved the lowest MAE at about $371,849, but its R² score of 0.034 was still below the Linear Regression baseline.

These results suggest that the current tree-based models did not outperform the baseline by the main Week 5 metric, R². However, the Random Forest’s lower MAE suggests it may be making better average predictions for many homes. Future work should include tuning tree model parameters, exploring additional feature engineering, and evaluating model performance across different price ranges to better understand where each model performs well or poorly.